# Fruit and Vegetable Fresh/Rotten Classification CNN.

## Tasks.
- Download, unzip and prepare dataset.
- Preprocess dataset.
- Train and test model.
- Implement Grading as set out in the case study.
- Export model.

In [48]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import random
import tensorflow as tf

### Checking if the dataset exists, if it does not then download it from custom S3 bucket and prepare it.
For simplicity the kaggle dataset was uploaded to an S3 bucket for ease-of-access.

In [49]:
DATASET_DIRECTORY = '../dataset/Fruit And Vegetable Diseases Dataset'
BATCH_SIZE = 32
IMAGE_SIZE = (244, 244)
EPOCHS = 50
RAN_SEED = 181181 # Arbitrary seed for random number generation.

# Set seeds for random number generators.
tf.random.set_seed(RAN_SEED)
np.random.seed(RAN_SEED)
random.seed(RAN_SEED)

# Checking if dataset is present, if not then download and extract.
def check_dataset():
    if not os.path.exists(DATASET_DIRECTORY):
        print('Dataset not found, checking directory...')
        if not os.path.isfile('../dataset/archive.zip'):
            print('Downloading archive...')
            import boto3
            from botocore import UNSIGNED
            from botocore.config import Config
            bucket_name = 'aai-university-content'
            endpoint = 'https://s3.fr-par.scw.cloud'
            file_key = 'archive.zip'
            local_file = '../dataset/archive.zip'
            s3 = boto3.client('s3',endpoint_url=endpoint,config=Config(signature_version=UNSIGNED))
            s3.download_file(bucket_name, file_key, local_file)
            print('Archive download complete.')

        print('Archive found, unzipping archive...')
        from zipfile import ZipFile
        with ZipFile("../dataset/archive.zip", 'r') as zip_ref:
            zip_ref.extractall(path="../dataset")
        print('Archive extraction complete.')
    print('Dataset found.')

check_dataset()

Dataset not found, checking directory...
Archive download complete.
Archive found, unzipping archive...
Archive extraction complete.
Dataset found.


### Identify classes, images and labels from dataset.


In [51]:
dataset_imgs = []
dataset_labels = []

# Identify the class names.
dataset_classes = sorted([x for x in os.listdir(DATASET_DIRECTORY) if os.path.isdir(os.path.join(DATASET_DIRECTORY, x))])

# Iterate through files to identify images and labels.
for label, dataset_class in enumerate(dataset_classes):
    directory = os.path.join(DATASET_DIRECTORY, dataset_class)
    for file_identifier in os.listdir(directory):
        if file_identifier.lower().endswith(('.jpg', '.jpeg', '.png')):
            dataset_imgs.append(os.path.join(directory, file_identifier))
            dataset_labels.append(label)

print("Classes loaded: " + str(len(dataset_classes)))
print("Images loaded: " + str(len(dataset_imgs)))
print("Labels loaded: " + str(len(dataset_labels)))

# Create simple table for displaying the total amount of data in each class.
dataset_label_ints = pd.Series(dataset_labels).value_counts().sort_index()
dataset_distribution = pd.DataFrame({
    "Class": dataset_classes,
    "Amount": dataset_label_ints.values
})
print("\nClass Totals: \n" + str(dataset_distribution))

Classes loaded: 28
Images loaded: 29277
Labels loaded: 29277

Class Totals: 
                   Class  Amount
0         Apple__Healthy    2438
1          Apple__Rotten    2925
2        Banana__Healthy    1999
3         Banana__Rotten    2797
4    Bellpepper__Healthy     611
5     Bellpepper__Rotten     591
6        Carrot__Healthy     619
7         Carrot__Rotten     579
8      Cucumber__Healthy     608
9       Cucumber__Rotten     593
10        Grape__Healthy     200
11         Grape__Rotten     200
12        Guava__Healthy     200
13         Guava__Rotten     200
14       Jujube__Healthy     200
15        Jujube__Rotten     200
16        Mango__Healthy    1813
17         Mango__Rotten    2247
18       Orange__Healthy    2075
19        Orange__Rotten    2186
20  Pomegranate__Healthy     200
21   Pomegranate__Rotten     200
22       Potato__Healthy     614
23        Potato__Rotten     584
24   Strawberry__Healthy    1603
25    Strawberry__Rotten    1596
26       Tomato__Healthy     604

### Split the data into training, validation and testing.